# GraphRAG com Entidades Melhoradas

Duas abordagens para melhorar a extracção de entidades, ambas gratuitas:

| Abordagem | Tempo | GPU | Qualidade esperada |
|---|---|---|---|
| **A — Léxico expandido + lemmatização** | ~5 min | Não | Moderada |
| **B — Qwen3-8B local** | ~1-2h | Sim (1 GPU) | Alta |

**Recomendado:** correr A primeiro (rápido), depois B em paralelo (melhor qualidade).

Se tiveres um batch Anthropic pendente, cancela-o na **Célula 0** antes de continuar.

In [ ]:
# ── Célula 0 — Cancelar batch Anthropic pendente (só se necessário) ──
import anthropic, os, getpass

BATCH_ID = "msgbatch_0195V96yixgytGpkbwQ4AXi5"

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")

client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
result = client.messages.batches.cancel(BATCH_ID)
print("Status após cancelamento:", result.processing_status)

In [ ]:
# ── Célula 1 — Setup (correr sempre primeiro) ─────────────────────────
import os, sys
from pathlib import Path

ROOT = "/home/jovyan/privado/LLMDermato/derm-rag"

os.environ["HF_HOME"] = "/home/jovyan/privado/.cache/huggingface"

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)

print(f"ROOT   : {ROOT}")
print(f"cwd    : {os.getcwd()}")
print(f"Python : {sys.executable}")

---
## Abordagem A — Léxico expandido + lemmatização (~5 min, sem GPU)

In [ ]:
# ── A1 — Extrair entidades com léxico expandido ───────────────────────
import importlib
import scripts.extract_entities_local as _local
importlib.reload(_local)

sys.argv = ["extract_entities_local.py", "--config", f"{ROOT}/configs/graph_lemmatized.yaml"]
_local.main()

In [ ]:
# ── A2 — Graph puro (lemmatizado) ─────────────────────────────────────
os.environ["CUDA_VISIBLE_DEVICES"] = "7"
import importlib, scripts.run_graph as _graph
importlib.reload(_graph)

sys.argv = ["run_graph.py", "--config", f"{ROOT}/configs/graph_lemmatized.yaml"]
_graph.main()

In [ ]:
# ── A3 — Graph + Dense híbrido (lemmatizado + BGE-M3 ft-v2) ──────────
import importlib, scripts.run_graph_hybrid as _ghybrid
importlib.reload(_ghybrid)

sys.argv = ["run_graph_hybrid.py", "--config", f"{ROOT}/configs/graph_hybrid_lemmatized.yaml"]
_ghybrid.main()

---
## Abordagem B — Qwen3-8B local (~1-2h, 1 GPU)

Usa o mesmo `Qwen/Qwen3-8B` já em cache no servidor.
Guarda checkpoint a cada 500 casos — se o kernel cair, retoma automaticamente.

In [ ]:
# ── B1 — Extrair entidades com Qwen3-8B ──────────────────────────────
os.environ["CUDA_VISIBLE_DEVICES"] = "7"   # ajusta para a GPU disponível

import importlib
import scripts.extract_entities_qwen as _qwen
importlib.reload(_qwen)

sys.argv = [
    "extract_entities_qwen.py",
    "--config",     f"{ROOT}/configs/graph_qwen.yaml",
    "--batch-size", "16",
]
_qwen.main()

In [ ]:
# ── B2 — Verificar cobertura das entidades Qwen ───────────────────────
import json, random

ent_path = Path(f"{ROOT}/data/processed/entities_qwen.json")
assert ent_path.exists(), f"Não encontrado: {ent_path}"

entities = json.loads(ent_path.read_text())
n = len(entities)
print(f"Total casos: {n}\n")

for key in ("disease", "morphology", "location", "symptom", "chemical"):
    has = sum(1 for e in entities.values() if e.get(key))
    avg = sum(len(e.get(key, [])) for e in entities.values()) / n
    print(f"  {key:<12}: {has:>5}/{n} ({has/n*100:.0f}%)  média {avg:.1f}/caso")

print("\n── 3 exemplos aleatórios ──")
for sid in random.sample(list(entities.keys()), 3):
    print(f"\n{sid}:")
    for k, v in entities[sid].items():
        if v: print(f"  {k}: {v}")

In [ ]:
# ── B3 — Graph puro (entidades Qwen) ─────────────────────────────────
import importlib, scripts.run_graph as _graph
importlib.reload(_graph)

sys.argv = ["run_graph.py", "--config", f"{ROOT}/configs/graph_qwen.yaml"]
_graph.main()

In [ ]:
# ── B4 — Graph + Dense híbrido (Qwen entities + BGE-M3 ft-v2) ────────
import importlib, scripts.run_graph_hybrid as _ghybrid
importlib.reload(_ghybrid)

sys.argv = ["run_graph_hybrid.py", "--config", f"{ROOT}/configs/graph_hybrid_qwen_entities.yaml"]
_ghybrid.main()

---
## Resultados — Tabela Comparativa

In [ ]:
# ── Tabela comparativa (correr após qualquer das abordagens) ──────────
import json
from pathlib import Path

results_dir = Path(f"{ROOT}/results")

def latest(prefix):
    files = [f for f in sorted(results_dir.glob(f"{prefix}*.json"),
                               key=lambda f: f.stat().st_mtime)
             if "rankings" not in f.name]
    return files[-1] if files else None

comparisons = [
    ("Graph entity (léxico fixo, original)",             "graphrag_entity"),
    ("Graph dense+RRF (original)",                        "graphrag_dense_rrf_qwen3_8b_finetuned_v2"),
    ("[A] Graph entity (lemmatizado)",                    "graphrag_lemmatized"),
    ("[A] Graph híbrido (lemmatizado + BGE-M3 ft-v2)",   "graphrag_hybrid_lemmatized_bge_m3_ft_v2"),
    ("[B] Graph entity (Qwen3-8B)",                       "graphrag_qwen_entities"),
    ("[B] Graph híbrido (Qwen3-8B + BGE-M3 ft-v2)",      "graphrag_hybrid_qwen_entities_bge_m3_ft_v2"),
    ("CRAG BGE-M3 ft-v2 + reranker ★ (melhor actual)",   "crag_v2_reranker_finetuned"),
]

print(f"{'Modelo':<52} {'R@1':>7} {'R@5':>7} {'R@10':>7} {'MRR':>7}")
print("-" * 79)
for label, prefix in comparisons:
    f = latest(prefix)
    if f is None:
        print(f"{label:<52} {'(sem resultado)'}")
        continue
    d = json.loads(f.read_text())
    m = d.get("metrics", d)
    def fmt(k): return f"{m[k]:.4f}" if k in m else "  —  "
    print(f"{label:<52} {fmt('recall@1'):>7} {fmt('recall@5'):>7} {fmt('recall@10'):>7} {fmt('mrr'):>7}")